#AZURE DEMAND FORECAST MODEL

In [ ]:
# all imports lib
%pip install statsmodels
%pip install xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from statsmodels.tsa.arima.model import ARIMA
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


#MileStone 1 (collection of datasets)

In [3]:
#loading of dataset
df = pd.read_csv("azure_demand_dataset.csv")
print(df)

       timestamp         region service_type  demand  capacity_allocation  \
0     01-01-2024      East Asia      Compute   562.0                754.0   
1     01-01-2024      East Asia      Storage   798.0               1062.0   
2     01-01-2024      East Asia      Network   351.0                408.0   
3     01-01-2024    South India      Compute   693.0                931.0   
4     01-01-2024    South India      Storage   406.0                524.0   
...          ...            ...          ...     ...                  ...   
5395  26-10-2024  Korea Central      Storage   353.0                426.0   
5396  26-10-2024  Korea Central      Network   194.0                261.0   
5397  26-10-2024  Central India      Compute   502.0                572.0   
5398  26-10-2024  Central India      Storage   510.0                628.0   
5399  26-10-2024  Central India      Network   501.0                615.0   

      cost_usd  service_availability_pct  is_holiday  economic_index  \
0  

In [4]:
#correcting the date and time format
df['timestamp'] = pd.to_datetime(df['timestamp'], format ='mixed')
df = df.sort_values(by='timestamp')
df = pd.DataFrame(df)
df

,timestamp,region,service_type,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index
0,2024-01-01,East Asia,Compute,562.0,754.0,148.49,99.88,1,93.51,3.38,63.64
16,2024-01-01,Central India,Storage,726.0,947.0,194.60,99.49,1,93.51,3.38,63.64
15,2024-01-01,Central India,Compute,NaN,716.0,139.36,99.48,1,93.51,3.38,63.64
14,2024-01-01,Korea Central,Network,664.0,850.0,115.31,99.54,1,93.51,3.38,63.64
13,2024-01-01,Korea Central,Storage,263.0,338.0,69.69,98.89,1,93.51,3.38,63.64
...,...,...,...,...,...,...,...,...,...,...,...
5134,2024-12-10,South India,Storage,170.0,203.0,44.75,NaN,0,84.78,1.87,76.69
5133,2024-12-10,South India,Compute,396.0,473.0,70.58,99.33,0,84.78,1.87,76.69
5132,2024-12-10,East Asia,Network,350.0,450.0,87.93,99.16,0,84.78,1.87,76.69
5139,2024-12-10,Southeast Asia,Compute,175.0,221.0,34.48,99.20,0,84.78,1.87,76.69


In [5]:
#region correction
df['region'] = df['region'].str.title().str.replace(' ','-')
df['region']


0            East-Asia
16       Central-India
15       Central-India
14       Korea-Central
13       Korea-Central
             ...      
5134       South-India
5133       South-India
5132         East-Asia
5139    Southeast-Asia
5145     Central-India
Name: region, Length: 5400, dtype: object

In [6]:
#duplicates and null values
print(df.duplicated())
print(df.isnull().sum())
df=df.drop_duplicates()
df.isnull().sum()

0       False
16      False
15      False
14      False
13      False
        ...  
5134    False
5133    False
5132    False
5139    False
5145    False
Length: 5400, dtype: bool
timestamp                     0
region                        0
service_type                  0
demand                      162
capacity_allocation         162
cost_usd                    162
service_availability_pct    162
is_holiday                    0
economic_index              162
market_growth_rate          162
cloud_adoption_index        162
dtype: int64


timestamp                     0
region                        0
service_type                  0
demand                      162
capacity_allocation         162
cost_usd                    162
service_availability_pct    162
is_holiday                    0
economic_index              162
market_growth_rate          162
cloud_adoption_index        162
dtype: int64

In [7]:
#column fills 
df['demand'] = df['demand'].fillna(df['demand'].mean())
df['capacity_allocation']=df['capacity_allocation'].fillna(df['capacity_allocation'].mean())
df['cost_usd']=df['cost_usd'].fillna(df['capacity_allocation']*0.5)
df['service_availability_pct'] = df['service_availability_pct'].ffill().bfill()
df['is_holiday'] = df['is_holiday'].fillna(0)
df['economic_index'] = df['economic_index'].fillna(df['economic_index'].mean())
df['market_growth_rate'] = df['market_growth_rate'].fillna(df['market_growth_rate'].mean())
df['cloud_adoption_index'] = df['cloud_adoption_index'].fillna(df['cloud_adoption_index'].mean())

In [8]:
#checking for null values
df.isnull().sum()

timestamp                   0
region                      0
service_type                0
demand                      0
capacity_allocation         0
cost_usd                    0
service_availability_pct    0
is_holiday                  0
economic_index              0
market_growth_rate          0
cloud_adoption_index        0
dtype: int64

In [9]:
#sort for use of lag , rolling amd spike
df = df.sort_values('timestamp').reset_index(drop='true')
print(df)

      timestamp          region service_type      demand  capacity_allocation  \
0    2024-01-01       East-Asia      Compute  562.000000           754.000000   
1    2024-01-01      Japan-East      Network  519.727377           835.000000   
2    2024-01-01       East-Asia      Storage  798.000000          1062.000000   
3    2024-01-01       East-Asia      Network  351.000000           408.000000   
4    2024-01-01     South-India      Compute  693.000000           931.000000   
...         ...             ...          ...         ...                  ...   
5395 2024-12-10   Central-India      Network  165.000000           209.000000   
5396 2024-12-10       East-Asia      Storage  597.000000           714.000000   
5397 2024-12-10  Southeast-Asia      Compute  175.000000           221.000000   
5398 2024-12-10       East-Asia      Compute  580.000000           664.000000   
5399 2024-12-10   Central-India      Compute  562.000000           646.342115   

      cost_usd  service_ava

In [10]:
#for seasonal pattern

df['days'] = df['timestamp'].dt.day
df['week'] = df['timestamp'].dt.weekday
df

,timestamp,region,service_type,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,week
0,2024-01-01,East-Asia,Compute,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,0
1,2024-01-01,Japan-East,Network,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,0
2,2024-01-01,East-Asia,Storage,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,0
3,2024-01-01,East-Asia,Network,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,0
4,2024-01-01,South-India,Compute,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,Central-India,Network,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,1
5396,2024-12-10,East-Asia,Storage,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,1
5397,2024-12-10,Southeast-Asia,Compute,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,1
5398,2024-12-10,East-Asia,Compute,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,1


In [11]:
#for adding lag (current compared to the just past daata)
df['lag1_sort_term'] = df['demand'].shift(1)
df['lag7_long_term_week'] = df['demand'].shift(7)
df

,timestamp,region,service_type,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,week,lag1_sort_term,lag7_long_term_week
0,2024-01-01,East-Asia,Compute,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,0,NaN,NaN
1,2024-01-01,Japan-East,Network,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,0,562.000000,NaN
2,2024-01-01,East-Asia,Storage,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,0,519.727377,NaN
3,2024-01-01,East-Asia,Network,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,0,798.000000,NaN
4,2024-01-01,South-India,Compute,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,0,351.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,Central-India,Network,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,1,443.000000,179.0
5396,2024-12-10,East-Asia,Storage,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,1,165.000000,305.0
5397,2024-12-10,Southeast-Asia,Compute,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,1,597.000000,550.0
5398,2024-12-10,East-Asia,Compute,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,1,175.000000,334.0


In [12]:
#adding rolling mean (avg of 3 consicutive datas)
df['rolling_mean_3'] = df['demand'].rolling(window=3).mean()
df

,timestamp,region,service_type,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,week,lag1_sort_term,lag7_long_term_week,rolling_mean_3
0,2024-01-01,East-Asia,Compute,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,0,NaN,NaN,NaN
1,2024-01-01,Japan-East,Network,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,0,562.000000,NaN,NaN
2,2024-01-01,East-Asia,Storage,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,0,519.727377,NaN,626.575792
3,2024-01-01,East-Asia,Network,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,0,798.000000,NaN,556.242459
4,2024-01-01,South-India,Compute,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,0,351.000000,NaN,614.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,Central-India,Network,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,1,443.000000,179.0,373.333333
5396,2024-12-10,East-Asia,Storage,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,1,165.000000,305.0,401.666667
5397,2024-12-10,Southeast-Asia,Compute,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,1,597.000000,550.0,312.333333
5398,2024-12-10,East-Asia,Compute,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,1,175.000000,334.0,450.666667


In [13]:
#flagging of unusal spike
threshold = df['demand'].mean() + df['demand'].std() 
df['spike'] = np.where(df['demand'] > threshold, 1, 0)
df

,timestamp,region,service_type,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,week,lag1_sort_term,lag7_long_term_week,rolling_mean_3,spike
0,2024-01-01,East-Asia,Compute,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,0,NaN,NaN,NaN,0
1,2024-01-01,Japan-East,Network,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,0,562.000000,NaN,NaN,0
2,2024-01-01,East-Asia,Storage,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,0,519.727377,NaN,626.575792,1
3,2024-01-01,East-Asia,Network,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,0,798.000000,NaN,556.242459,0
4,2024-01-01,South-India,Compute,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,0,351.000000,NaN,614.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,Central-India,Network,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,1,443.000000,179.0,373.333333,0
5396,2024-12-10,East-Asia,Storage,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,1,165.000000,305.0,401.666667,0
5397,2024-12-10,Southeast-Asia,Compute,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,1,597.000000,550.0,312.333333,0
5398,2024-12-10,East-Asia,Compute,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,1,175.000000,334.0,450.666667,0


In [14]:
#convertion to one hot encoding 

df = pd.get_dummies(df, columns=['region','service_type'],drop_first = True)
df

,timestamp,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,...,lag7_long_term_week,rolling_mean_3,spike,region_East-Asia,region_Japan-East,region_Korea-Central,region_South-India,region_Southeast-Asia,service_type_Network,service_type_Storage
0,2024-01-01,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,...,NaN,NaN,0,True,False,False,False,False,False,False
1,2024-01-01,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,...,NaN,NaN,0,False,True,False,False,False,True,False
2,2024-01-01,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,...,NaN,626.575792,1,True,False,False,False,False,False,True
3,2024-01-01,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,...,NaN,556.242459,0,True,False,False,False,False,True,False
4,2024-01-01,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,...,NaN,614.000000,0,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,...,179.0,373.333333,0,False,False,False,False,False,True,False
5396,2024-12-10,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,...,305.0,401.666667,0,True,False,False,False,False,False,True
5397,2024-12-10,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,...,550.0,312.333333,0,False,False,False,False,True,False,False
5398,2024-12-10,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,...,334.0,450.666667,0,True,False,False,False,False,False,False


In [15]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   timestamp                 5400 non-null   datetime64[ns]
 1   demand                    5400 non-null   float64       
 2   capacity_allocation       5400 non-null   float64       
 3   cost_usd                  5400 non-null   float64       
 4   service_availability_pct  5400 non-null   float64       
 5   is_holiday                5400 non-null   int64         
 6   economic_index            5400 non-null   float64       
 7   market_growth_rate        5400 non-null   float64       
 8   cloud_adoption_index      5400 non-null   float64       
 9   days                      5400 non-null   int32         
 10  week                      5400 non-null   int32         
 11  lag1_sort_term            5399 non-null   float64       
 12  lag7_long_term_week 

,timestamp,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,week,lag1_sort_term,lag7_long_term_week,rolling_mean_3,spike
count,5400,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5400.000000,5399.000000,5393.000000,5398.000000,5400.000000
mean,2024-06-10 07:31:12,519.727377,646.342115,135.572423,99.250798,0.536667,96.171126,3.304250,74.164099,15.136667,2.996667,519.719547,519.839391,519.713213,0.184630
min,2024-01-01 00:00:00,150.000000,170.000000,23.940000,98.500000,0.000000,83.370000,1.520000,60.030000,1.000000,0.000000,150.000000,150.000000,177.666667,0.000000
25%,2024-03-21 18:00:00,356.750000,440.000000,83.640000,98.870000,0.000000,89.120000,2.490000,66.720000,7.000000,1.000000,356.500000,357.000000,414.666667,0.000000
50%,2024-06-08 12:00:00,514.000000,636.000000,123.795000,99.250000,1.000000,97.310000,3.304250,74.164099,15.500000,3.000000,514.000000,514.000000,510.666667,0.000000
75%,2024-08-28 06:00:00,661.000000,822.000000,172.342500,99.630000,1.000000,102.610000,4.150000,81.830000,23.000000,5.000000,661.000000,661.000000,616.666667,0.000000
max,2024-12-10 00:00:00,999.000000,1377.000000,610.000000,99.990000,1.000000,106.880000,5.000000,89.850000,31.000000,6.000000,999.000000,999.000000,956.000000,1.000000
std,NaN,201.038034,255.711155,70.450789,0.432152,0.498700,7.168795,0.991474,8.572387,9.053535,2.009329,201.055832,201.047647,138.323969,0.388033


In [16]:
df.isnull().sum()

timestamp                   0
demand                      0
capacity_allocation         0
cost_usd                    0
service_availability_pct    0
is_holiday                  0
economic_index              0
market_growth_rate          0
cloud_adoption_index        0
days                        0
week                        0
lag1_sort_term              1
lag7_long_term_week         7
rolling_mean_3              2
spike                       0
region_East-Asia            0
region_Japan-East           0
region_Korea-Central        0
region_South-India          0
region_Southeast-Asia       0
service_type_Network        0
service_type_Storage        0
dtype: int64

#MileStone 3 using arima and xgboost to compare the results

In [17]:
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["day"] = df["timestamp"].dt.day
df

,timestamp,demand,capacity_allocation,cost_usd,service_availability_pct,is_holiday,economic_index,market_growth_rate,cloud_adoption_index,days,...,region_East-Asia,region_Japan-East,region_Korea-Central,region_South-India,region_Southeast-Asia,service_type_Network,service_type_Storage,year,month,day
0,2024-01-01,562.000000,754.000000,148.49,99.88,1,93.510000,3.38,63.64,1,...,True,False,False,False,False,False,False,2024,1,1
1,2024-01-01,519.727377,835.000000,221.40,99.83,1,93.510000,3.38,63.64,1,...,False,True,False,False,False,True,False,2024,1,1
2,2024-01-01,798.000000,1062.000000,183.12,99.56,1,93.510000,3.38,63.64,1,...,True,False,False,False,False,False,True,2024,1,1
3,2024-01-01,351.000000,408.000000,112.47,99.54,1,93.510000,3.38,63.64,1,...,True,False,False,False,False,True,False,2024,1,1
4,2024-01-01,693.000000,931.000000,192.24,99.38,1,93.510000,3.38,63.64,1,...,False,False,False,True,False,False,False,2024,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,2024-12-10,165.000000,209.000000,49.09,98.53,0,84.780000,1.87,76.69,10,...,False,False,False,False,False,True,False,2024,12,10
5396,2024-12-10,597.000000,714.000000,193.52,98.86,0,84.780000,1.87,76.69,10,...,True,False,False,False,False,False,True,2024,12,10
5397,2024-12-10,175.000000,221.000000,34.48,99.20,0,84.780000,1.87,76.69,10,...,False,False,False,False,True,False,False,2024,12,10
5398,2024-12-10,580.000000,664.000000,116.06,99.43,0,96.171126,1.87,76.69,10,...,True,False,False,False,False,False,False,2024,12,10


In [18]:
#After extracting useful features, the original timestamp column is no longer needed, so we drop it.

df = df.drop("timestamp", axis=1)

In [19]:
#x: feature which will predict the target (Y)

X = df.drop("demand", axis=1)
y = df["demand"]

In [20]:
#spliting the train and test data 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train)

      capacity_allocation  cost_usd  service_availability_pct  is_holiday  \
2028                417.0    208.50                     99.52           1   
282                 589.0    106.23                     98.70           1   
2146                918.0    127.13                     99.04           0   
3692                901.0    239.11                     98.99           0   
3659               1059.0    283.97                     99.44           1   
...                   ...       ...                       ...         ...   
871                 541.0     97.33                     99.11           0   
4764                281.0     53.05                     99.93           0   
4949                590.0     91.03                     98.62           1   
2925                604.0    154.81                     98.73           0   
3137                787.0    154.64                     99.09           1   

      economic_index  market_growth_rate  cloud_adoption_index  days  week 

In [21]:
#ARIMA MODEL(base model caputes time trends)

arima_model = ARIMA(y_train, order=(1,1,1)) #p,d,q
arima_model_fit = arima_model.fit()

arima_pred = arima_model_fit.forecast(steps=len(y_test))

arima_mse = np.sqrt(mean_squared_error(y_test, arima_pred))
print(arima_mse)

c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


203.00160358972272


c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [22]:
# XGBoost Model

xgboost_model = XGBRegressor(
    n_estimators=100, #100 trees
    max_depth=4, #tree depth 4 splits
    learning_rate=0.1,#how much each tree contributes
    random_state=42,
    objective="reg:squarederror"##define loss function used for regression
)

xgboost_model.fit(X_train, y_train)

xgboost_pred = xgboost_model.predict(X_test)

xgboost_mse = np.sqrt(mean_squared_error(y_test, xgboost_pred))
print(xgboost_mse)

46.26654024802925


In [23]:
print("Model Performance Comparison")
print("-----------------------------")
print(f"ARIMA Model RMSE   : {arima_mse:.4f}")
print(f"XGBoost Model RMSE : {xgboost_mse:.4f}")

Model Performance Comparison
-----------------------------
ARIMA Model RMSE   : 203.0016
XGBoost Model RMSE : 46.2665


In [24]:
# ARIMA Hyperparameter Tuning

p = range(0,4)#0 -> 3 lag
d = range(0,2)#0 -> 1 diffrence for the spike 
q = range(0,4)#0 -> 3 moving avg forecasting error or past error

#initial best score infinity
best_score = float("inf")
best_order = None

In [25]:
#searching for best combination to give less predictive error

for i in p: #for p 
    for j in d: #for d
        for k in q: #for q
            try:
                model = ARIMA(y_train, order=(i,j,k))
                model_fit = model.fit()
                pred = model_fit.forecast(steps=len(y_test))
                rmse = np.sqrt(mean_squared_error(y_test, pred))

                if rmse < best_score:
                    best_score = rmse
                    best_order = (i,j,k)

            except:
                continue

c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results 

In [26]:
# Train the Best ARIMA Model

best_arima_model = ARIMA(y_train, order=best_order).fit()
arima_tuned_pred = best_arima_model.forecast(steps=len(y_test))
arima_mse = np.sqrt(mean_squared_error(y_test , arima_tuned_pred))

c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
c:\Users\rites\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters

In [27]:
#hyperparameter tuning for xgboost model
param_grid = {
    "n_estimators": [50,100,200,500], # no of trees
    "max_depth": [2,3,4,5,6,7,8], # tree splits
    "learning_rate": [0.001,0.01,0.05,0.1,0.2],
    "subsample": [0.6,0.8,1],#how much sample data are use to train each tree
    "colsample_bytree": [0.7,0.8,1]#how many features are selected randomly to create the model
}

In [28]:
# GridSearchCV

grid_search = GridSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", random_state=42), #trainiing the base model
    param_grid=param_grid, #here all the combinations are saved in the above code
    scoring="neg_mean_squared_error", #gridsearch tries to maximize so sklearn keeps it negative higher negative mse = better 
    cv=3, 
    verbose=1 #printing the progress
)

grid_search.fit(X_train, y_train) #fitting the model

Fitting 3 folds for each of 1260 candidates, totalling 3780 fits


GridSearchCV(cv=3,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None...
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None,
                                    multi_strategy=None, n_estimators=None,
                                    n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'colsample_bytree': [0.7, 0.8, 1],
                         'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2],
                         'max_depth': [2, 3, 4, 5, 6, 7, 8],
                         'n_estimators': [50, 100, 200, 500],
                         'subsample': [0.6, 0.8, 1]},
             scoring='neg_mean_squared_error', verbose=1)

In [29]:
#choosing best xgboost and predicting

best_xgb = grid_search.best_estimator_ 
xgboost_tuned_pred = best_xgb.predict(X_test)

In [38]:
#1 vs 1 

final_arima = np.sqrt(mean_squared_error(y_test, arima_tuned_pred))
final_xgboost = np.sqrt(mean_squared_error(y_test, xgboost_tuned_pred))
print('        ------------------------------------------------------')
print("         Less Value Model Wins as it is Root Mean Square Error")
print('        ------------------------------------------------------')

print("ARIMA RMSE :", final_arima)
print("XGBoost RMSE:", final_xgboost)

        ------------------------------------------------------
         Less Value Model Wins as it is Root Mean Square Error
        ------------------------------------------------------
ARIMA RMSE : 202.9184301540681
XGBoost RMSE: 45.456816298709214
